# 자산현황 입력 및 조회

## 0. 초기화

In [1]:
import pandas as pd
import sqlite3
import ledgerly.utils as utils


conn = sqlite3.connect(utils.find_project_root() / 'data' / 'db' / 'ledgerly.db')


In [2]:
account_path = utils.find_project_root() / 'data' / 'raw' / 'asset_account.csv'
snapshot_path = utils.find_project_root() / 'data' / 'raw' / 'asset_snapshot.csv'

In [3]:
account_df = pd.read_csv(account_path)
snapshot_df = pd.read_csv(snapshot_path)

# 1. asset account 입력

In [4]:
account_df.head()

,ID,자산이름,분류,소유주,비고
0,진영_카뱅_통장,진영 카뱅 통장,자율예금,진영,NaN
1,진영_자율_통장,진영 자율 통장,자율예금,진영,NaN
2,준열_생활비통장,생활비통장,자율예금,준열,NaN
3,준열_카뱅통장,카뱅통장,자율예금,준열,NaN
4,준열_기타통장,기타통장,자율예금,준열,NaN


In [5]:
account_df = account_df.rename(columns={
     "ID": "asset_id",
    "자산이름": "asset_name",
    "분류": "category",
    "소유주": "owner",
    "비고": "memo"
})

In [6]:
account_df.head()

,asset_id,asset_name,category,owner,memo
0,진영_카뱅_통장,진영 카뱅 통장,자율예금,진영,NaN
1,진영_자율_통장,진영 자율 통장,자율예금,진영,NaN
2,준열_생활비통장,생활비통장,자율예금,준열,NaN
3,준열_카뱅통장,카뱅통장,자율예금,준열,NaN
4,준열_기타통장,기타통장,자율예금,준열,NaN


In [7]:
account_df = account_df.where(pd.notnull(account_df), None )

In [12]:
account_df.head()

,asset_id,asset_name,category,owner,memo
0,진영_카뱅_통장,진영 카뱅 통장,자율예금,진영,NaN
1,진영_자율_통장,진영 자율 통장,자율예금,진영,NaN
2,준열_생활비통장,생활비통장,자율예금,준열,NaN
3,준열_카뱅통장,카뱅통장,자율예금,준열,NaN
4,준열_기타통장,기타통장,자율예금,준열,NaN


In [10]:
upsert_sql = """
INSERT INTO asset_account (
    asset_id,
    asset_name,
    category,
    owner,
    memo,
    created_at,
    updated_at
)
VALUES (
    :asset_id,
    :asset_name,
    :category,
    :owner,
    :memo,
    datetime('now'),
    datetime('now')
)
ON CONFLICT(asset_id) DO UPDATE SET
    asset_name = excluded.asset_name,
    category   = excluded.category,
    owner      = excluded.owner,
    memo       = excluded.memo,
    updated_at = datetime('now');
"""

In [15]:
cursor = conn.cursor()
for _, row in account_df.iterrows():
    print(row)
    cursor.execute(upsert_sql, {
        "asset_id": row["asset_id"],
        "asset_name": row["asset_name"],
        "category": row["category"],
        "owner": row["owner"],
        "memo": row["memo"]
    })
conn.commit()

asset_id      진영_카뱅_통장
asset_name    진영 카뱅 통장
category          자율예금
owner               진영
memo               NaN
Name: 0, dtype: object
asset_id      진영_자율_통장
asset_name    진영 자율 통장
category          자율예금
owner               진영
memo               NaN
Name: 1, dtype: object
asset_id      준열_생활비통장
asset_name       생활비통장
category          자율예금
owner               준열
memo               NaN
Name: 2, dtype: object
asset_id      준열_카뱅통장
asset_name       카뱅통장
category         자율예금
owner              준열
memo              NaN
Name: 3, dtype: object
asset_id      준열_기타통장
asset_name       기타통장
category         자율예금
owner              준열
memo              NaN
Name: 4, dtype: object
asset_id      준열_맑은하늘적금
asset_name       맑은하늘적금
category             적금
owner                준열
memo                NaN
Name: 5, dtype: object
asset_id      준열_주택청약
asset_name    준열 주택청약
category           청약
owner              준열
memo              NaN
Name: 6, dtype: object
asset_id      진영_주택청약
asset_name    진영 주택청약


## 2. snapshot 입력

In [27]:
snapshot_df = pd.read_csv(snapshot_path)

In [20]:
snapshot_insert_sql = """
insert into asset_snapshot (
    asset_id,
    snapshot_date,
    amount,
    rate
) values (
    :asset_id, 
    :snapshot_date,
    :amount,
    :rate
    );
    """
    
    
    

In [21]:
snapshot_df.head()

,ID,정산일,금액,이율/수익률
0,진영_카뱅_통장,2026-02-24,508446,NaN
1,진영_자율_통장,2026-02-24,9912,NaN
2,준열_생활비통장,2026-02-24,2490870,NaN
3,준열_카뱅통장,2026-02-24,345000,NaN
4,준열_기타통장,2026-02-24,8784,NaN


In [22]:
snapshot_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ID      24 non-null     object 
 1   정산일     24 non-null     object 
 2   금액      24 non-null     int64  
 3   이율/수익률  1 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 900.0+ bytes


In [28]:
snapshot_df = snapshot_df.rename(columns={
    "ID": "asset_id",
    "정산일": "snapshot_date",
    "금액": "amount",
    "이율/수익률" : "rate"
})

snapshot_df.head()    

,asset_id,snapshot_date,amount,rate
0,진영_카뱅_통장,2026-01-31,7908,NaN
1,진영_자율_통장,2026-01-31,18409,NaN
2,준열_생활비통장,2026-01-31,2329808,NaN
3,준열_카뱅통장,2026-01-31,373856,NaN
4,준열_기타통장,2026-01-31,8784,NaN


In [29]:
for _, row in snapshot_df.iterrows():
    print(row)
    cursor.execute(snapshot_insert_sql, {
        "asset_id": row["asset_id"],
        "snapshot_date": row["snapshot_date"],
        "amount": row["amount"],
        "rate": row.get("rate", None)
    })
    
conn.commit()


asset_id           진영_카뱅_통장
snapshot_date    2026-01-31
amount                 7908
rate                    NaN
Name: 0, dtype: object
asset_id           진영_자율_통장
snapshot_date    2026-01-31
amount                18409
rate                    NaN
Name: 1, dtype: object
asset_id           준열_생활비통장
snapshot_date    2026-01-31
amount              2329808
rate                    NaN
Name: 2, dtype: object
asset_id            준열_카뱅통장
snapshot_date    2026-01-31
amount               373856
rate                    NaN
Name: 3, dtype: object
asset_id            준열_기타통장
snapshot_date    2026-01-31
amount                 8784
rate                    NaN
Name: 4, dtype: object
asset_id          준열_맑은하늘적금
snapshot_date    2026-01-31
amount              3900000
rate                    NaN
Name: 5, dtype: object
asset_id            준열_주택청약
snapshot_date    2026-01-31
amount              7400000
rate                    NaN
Name: 6, dtype: object
asset_id            진영_주택청약
snapshot_date    2026-01-31